In [33]:
from ultralytics import YOLO
import cv2
import numpy as np
import math

# YOLO modelini yükle
model = YOLO("yolov8n.pt")

# Video dosyasını aç
cap = cv2.VideoCapture("araba.mp4")

# FPS ve boyut bilgilerini al
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# FPS geçerli değilse manuel düzelt
if fps <= 0 or fps > 120:
    print("⚠️ FPS değeri hatalı okundu, varsayılan 30 olarak ayarlanıyor.")
    fps = 30.0

print(f"Video boyutu: {width}x{height}")
print(f"FPS: {fps}")

# Görüntüleme boyutu
display_width = 960
display_height = 720

# --- Perspektif bölge noktaları ---
tl = (1062, 972)
bl = (1700, 972)
tr = (50, 1800)
br = (1730, 1800)
region_pts = np.array([tl, bl, br, tr], np.int32).reshape((-1, 1, 2))

# --- Perspektif dönüşümü hedef boyutu ---
bird_width, bird_height = 960, 720
src_pts = np.float32([tl, bl, br, tr])
dst_pts = np.float32([[0, 0], [0, bird_height], [bird_width, bird_height], [bird_width, 0]])
M = cv2.getPerspectiveTransform(src_pts, dst_pts)

# 7.5 m genişlik referansı
real_width_m = 7.5
pixel_width = math.dist(bl, tl)
px_to_m = real_width_m / pixel_width

# Takip ve sayaçlar
track_history = {}
object_positions = {}
entered_ids = set()
total_entered = 0
object_id_counter = 0

frame_index = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break
    frame_index += 1

    # YOLO tespiti (araç sınıfları)
    results = model(frame, classes=[2, 3, 5, 7])
    detections = results[0].boxes.data.cpu().numpy()

    annotated_frame = frame.copy()

    # Bölgeyi çiz
    cv2.polylines(annotated_frame, [region_pts], True, (0, 255, 0), 2)
    overlay = annotated_frame.copy()
    cv2.fillPoly(overlay, [region_pts], color=(0, 255, 0))
    annotated_frame = cv2.addWeighted(overlay, 0.2, annotated_frame, 0.8, 0)

    new_positions = {}

    for det in detections:
        x1, y1, x2, y2, conf, cls = det
        cx, cy = int((x1 + x2) / 2), int((y1 + y2) / 2)
        center = (cx, cy)

        inside = cv2.pointPolygonTest(region_pts, center, False)
        matched_id = None

        # ID eşleştirme
        for oid, (px, py) in object_positions.items():
            if math.dist((px, py), (cx, cy)) < 60:
                matched_id = oid
                break

        if matched_id is None:
            object_id_counter += 1
            matched_id = object_id_counter

        new_positions[matched_id] = (cx, cy)

        # Alan giriş kontrolü
        if inside > 0 and matched_id not in entered_ids:
            total_entered += 1
            entered_ids.add(matched_id)

        # Sadece alan içindeyse hız hesapla
        if inside > 0:
            if matched_id not in track_history:
                track_history[matched_id] = []
            track_history[matched_id].append((cx, cy, frame_index))

            # Hız
            speed_text = ""
            if len(track_history[matched_id]) >= 2:
                (x1p, y1p, f1), (x2p, y2p, f2) = track_history[matched_id][-2], track_history[matched_id][-1]
                pixel_dist = math.dist((x1p, y1p), (x2p, y2p))
                time_diff = (f2 - f1) / fps  # FPS hesaba katıldı
                if time_diff > 0:
                    speed_m_s = (pixel_dist * px_to_m) / time_diff
                    speed_kmh = speed_m_s * 3.6
                    speed_text = f"{speed_kmh:.1f} km/h"
            else:
                speed_text = "..."

            # Görsel
            color = (0, 255, 255)
            cv2.rectangle(annotated_frame, (int(x1), int(y1)), (int(x2), int(y2)), color, 2)
            cv2.putText(annotated_frame, f"ID {matched_id}", (int(x1), int(y1) - 45),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
            cv2.putText(annotated_frame, speed_text, (int(x1), int(y1) - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 255), 3)

    object_positions = new_positions

    # Sayaç
    cv2.putText(annotated_frame, f"Total Vehicles Entered: {total_entered}",
                (60, 80), cv2.FONT_HERSHEY_SIMPLEX, 1.4, (0, 255, 0), 4)

    # Perspektif görünüm (bird-eye)
    bird_eye = cv2.warpPerspective(frame, M, (bird_width, bird_height))
    cv2.putText(bird_eye, "Perspective View", (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    # Görüntüleri ayrı pencerelerde göster
    resized_main = cv2.resize(annotated_frame, (display_width, display_height))
    resized_bird = cv2.resize(bird_eye, (display_width, display_height))

    cv2.imshow("YOLO Speed View", resized_main)
    cv2.imshow("Perspective (Bird-Eye) View", resized_bird)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


Video boyutu: 3840x2160
FPS: 25.0

0: 384x640 3 cars, 1 truck, 49.6ms
Speed: 2.3ms preprocess, 49.6ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 4 cars, 1 truck, 56.6ms
Speed: 2.5ms preprocess, 56.6ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 4 cars, 1 bus, 1 truck, 54.9ms
Speed: 18.4ms preprocess, 54.9ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 4 cars, 1 bus, 1 truck, 105.7ms
Speed: 3.0ms preprocess, 105.7ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 cars, 1 truck, 54.5ms
Speed: 1.8ms preprocess, 54.5ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 cars, 1 bus, 1 truck, 49.7ms
Speed: 2.6ms preprocess, 49.7ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 cars, 1 truck, 86.6ms
Speed: 2.0ms preprocess, 86.6ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640